# Lab 4 - Homografia e Panorama

**Objetivo:** Capturar 3 imagens de uma mesma cena em angulos/posicoes levemente
diferentes e realizar a combinacao dessas imagens para montar um panorama
utilizando homografia.

**Pipeline:**
1. Captura / upload das 3 imagens
2. Deteccao de keypoints com SIFT
3. Correspondencia de features entre imagens consecutivas
4. Calculo das matrizes de homografia via RANSAC
5. Warping e composicao do panorama

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f'OpenCV versao: {cv2.__version__}')

## 1. Carregamento das imagens

As imagens devem estar em uma pasta chamada **`imagens/`** com os nomes
**`esquerda`**, **`centro`** e **`direita`** (qualquer extensao: `.jpg`, `.jpeg`, `.png`).

**No Google Colab**, faca o upload da pasta `imagens` (ou crie a pasta e
arraste as 3 fotos) para o diretorio de trabalho `/content/`. A celula abaixo
tambem oferece upload manual caso a pasta nao seja encontrada.

In [ ]:
import glob

PASTA_IMAGENS = 'imagens'


def localizar_imagem(pasta, nome):
    """Procura um arquivo 'nome.*' (qualquer extensao) dentro de 'pasta'."""
    padroes = glob.glob(os.path.join(pasta, nome + '.*'))
    if not padroes:
        raise FileNotFoundError(
            f"Nao encontrei a imagem '{nome}' em '{pasta}/'. "
            f"Arquivos esperados: {nome}.jpg / {nome}.jpeg / {nome}.png")
    return padroes[0]


def carregar_imagem(caminho):
    img = cv2.imread(caminho)
    if img is None:
        raise ValueError(f'Falha ao ler a imagem: {caminho}')
    return img


# No Colab: se a pasta nao existir, oferece upload manual das 3 imagens.
if not os.path.isdir(PASTA_IMAGENS):
    try:
        from google.colab import files
        print(f"Pasta '{PASTA_IMAGENS}/' nao encontrada.")
        print('Selecione 3 imagens nomeadas esquerda, centro e direita:')
        os.makedirs(PASTA_IMAGENS, exist_ok=True)
        enviados = files.upload()
        for nome_arq, conteudo in enviados.items():
            with open(os.path.join(PASTA_IMAGENS, nome_arq), 'wb') as f:
                f.write(conteudo)
    except ImportError:
        raise FileNotFoundError(
            f"Crie a pasta '{PASTA_IMAGENS}/' com as imagens "
            'esquerda, centro e direita.')

caminho_esq = localizar_imagem(PASTA_IMAGENS, 'esquerda')
caminho_cen = localizar_imagem(PASTA_IMAGENS, 'centro')
caminho_dir = localizar_imagem(PASTA_IMAGENS, 'direita')

# img1 = esquerda, img2 = centro (referencia), img3 = direita
img1 = carregar_imagem(caminho_esq)
img2 = carregar_imagem(caminho_cen)
img3 = carregar_imagem(caminho_dir)

print(f'Esquerda ({os.path.basename(caminho_esq)}): {img1.shape[1]}x{img1.shape[0]}')
print(f'Centro   ({os.path.basename(caminho_cen)}): {img2.shape[1]}x{img2.shape[0]}')
print(f'Direita  ({os.path.basename(caminho_dir)}): {img3.shape[1]}x{img3.shape[0]}')

### Pre-processamento: redimensionamento

In [ ]:
def redimensionar(img, escala=0.5):
    h, w = img.shape[:2]
    return cv2.resize(img, (int(w * escala), int(h * escala)))


ESCALA = 0.5  # ajuste se necessario
img1 = redimensionar(img1, ESCALA)
img2 = redimensionar(img2, ESCALA)
img3 = redimensionar(img3, ESCALA)

print(f'Apos redimensionamento (escala={ESCALA}):')
print(f'  img1: {img1.shape[1]}x{img1.shape[0]}')
print(f'  img2: {img2.shape[1]}x{img2.shape[0]}')
print(f'  img3: {img3.shape[1]}x{img3.shape[0]}')

### Visualizacao das 3 imagens capturadas

In [ ]:
def bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, titulo in zip(
        axes,
        [img1, img2, img3],
        ['Imagem 1 (Esquerda)', 'Imagem 2 (Centro)', 'Imagem 3 (Direita)']):
    ax.imshow(bgr2rgb(img))
    ax.set_title(titulo, fontsize=13)
    ax.axis('off')
plt.suptitle('Imagens capturadas', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Deteccao de Keypoints com SIFT

O **SIFT** (Scale-Invariant Feature Transform) detecta pontos de interesse robustos
a mudancas de escala, rotacao e iluminacao. Para cada keypoint e gerado um
descritor de 128 dimensoes.

In [ ]:
sift = cv2.SIFT_create()

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
gray3 = cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY)

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)
kp3, des3 = sift.detectAndCompute(gray3, None)

print('Keypoints detectados:')
print(f'  Imagem 1: {len(kp1)}')
print(f'  Imagem 2: {len(kp2)}')
print(f'  Imagem 3: {len(kp3)}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, kp, titulo in zip(
        axes, [img1, img2, img3], [kp1, kp2, kp3],
        ['Imagem 1', 'Imagem 2', 'Imagem 3']):
    img_kp = cv2.drawKeypoints(img, kp, None,
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    ax.imshow(bgr2rgb(img_kp))
    ax.set_title(f'{titulo} - {len(kp)} keypoints', fontsize=12)
    ax.axis('off')
plt.suptitle('Keypoints SIFT', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Correspondencia de Features

Usamos o **BFMatcher** (Brute-Force) com o **Lowe Ratio Test** (limiar 0.75)
para filtrar correspondencias ambiguas.

In [ ]:
def match_features(des_a, des_b, ratio=0.75):
    """Retorna os bons matches apos o ratio test de Lowe."""
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des_a, des_b, k=2)
    good = [m for m, n in matches if m.distance < ratio * n.distance]
    return good


matches_12 = match_features(des1, des2)
matches_32 = match_features(des3, des2)  # img3 -> img2 (referencia)

print('Matches aceitos (ratio test):')
print(f'  Imagem 1 -> Imagem 2: {len(matches_12)}')
print(f'  Imagem 3 -> Imagem 2: {len(matches_32)}')

img_match12 = cv2.drawMatchesKnn(
    img1, kp1, img2, kp2, [[m] for m in matches_12[:60]],
    None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
img_match32 = cv2.drawMatchesKnn(
    img3, kp3, img2, kp2, [[m] for m in matches_32[:60]],
    None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

fig, axes = plt.subplots(2, 1, figsize=(18, 10))
axes[0].imshow(bgr2rgb(img_match12))
axes[0].set_title(f'Matches: Imagem 1 -> Imagem 2  ({len(matches_12)} correspondencias)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(bgr2rgb(img_match32))
axes[1].set_title(f'Matches: Imagem 3 -> Imagem 2  ({len(matches_32)} correspondencias)', fontsize=12)
axes[1].axis('off')
plt.suptitle('Correspondencia de Features (BFMatcher + Ratio Test)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Calculo das Matrizes de Homografia

A **homografia** e uma transformacao projetiva 3x3 que mapeia pontos de uma
imagem para outra. E estimada com **RANSAC** para rejeitar correspondencias
incorretas (outliers).

In [ ]:
def calcular_homografia(matches, kp_src, kp_dst, min_matches=10):
    """
    Calcula a homografia que mapeia pontos de kp_src para kp_dst.
    Retorna (H, mask_inliers).
    """
    if len(matches) < min_matches:
        raise ValueError(f'Poucos matches: {len(matches)} (minimo: {min_matches})')
    pts_src = np.float32([kp_src[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts_dst = np.float32([kp_dst[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(pts_src, pts_dst, cv2.RANSAC, ransacReprojThreshold=5.0)
    return H, mask


# H_1to2: mapeia pontos da img1 para o espaco da img2
H_1to2, mask_12 = calcular_homografia(matches_12, kp1, kp2)
# H_3to2: mapeia pontos da img3 para o espaco da img2 (referencia central)
H_3to2, mask_32 = calcular_homografia(matches_32, kp3, kp2)

print('Homografia H (img1 -> img2):')
print(np.round(H_1to2, 4))
print(f'Inliers RANSAC: {int(mask_12.sum())}/{len(matches_12)}\n')

print('Homografia H (img3 -> img2):')
print(np.round(H_3to2, 4))
print(f'Inliers RANSAC: {int(mask_32.sum())}/{len(matches_32)}')

## 5. Construcao do Panorama

**Estrategia:**
- **Imagem 2** e a referencia (centro do panorama).
- A Imagem 1 e reprojetada no espaco de img2 usando `H_1to2`.
- A Imagem 3 e reprojetada no espaco de img2 usando `H_3to2`.
- Calcula-se o *bounding box* de todas as imagens transformadas para o canvas.
- Uma matriz de **translacao** `T` compensa coordenadas negativas.
- Composicao: img2 tem prioridade nas regioes de sobreposicao.

In [ ]:
def cantos(w, h):
    """Retorna os 4 cantos de uma imagem WxH no formato de perspectiveTransform."""
    return np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)


def criar_panorama(img1, img2, img3, H_1to2, H_3to2):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    h3, w3 = img3.shape[:2]

    # Transforma cantos de img1 e img3 para o espaco de img2
    c1 = cv2.perspectiveTransform(cantos(w1, h1), H_1to2)
    c2 = cantos(w2, h2)
    c3 = cv2.perspectiveTransform(cantos(w3, h3), H_3to2)

    todos = np.concatenate([c1, c2, c3], axis=0)
    min_x = int(np.floor(todos[:, 0, 0].min()))
    min_y = int(np.floor(todos[:, 0, 1].min()))
    max_x = int(np.ceil (todos[:, 0, 0].max()))
    max_y = int(np.ceil (todos[:, 0, 1].max()))

    # Translacao para evitar coordenadas negativas no canvas
    tx = max(0, -min_x)
    ty = max(0, -min_y)
    T  = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], dtype=np.float64)

    canvas_w = max_x - min_x
    canvas_h = max_y - min_y
    print(f'Canvas: {canvas_w}x{canvas_h}  |  translacao: ({tx}, {ty})')

    # Warping de cada imagem para o canvas
    img1_w = cv2.warpPerspective(img1, T @ H_1to2, (canvas_w, canvas_h))
    img2_w = cv2.warpPerspective(img2, T,           (canvas_w, canvas_h))
    img3_w = cv2.warpPerspective(img3, T @ H_3to2, (canvas_w, canvas_h))

    # Composicao: img1 e img3 preenchem o canvas; img2 tem prioridade
    mask1 = img1_w.sum(axis=2) > 0
    mask2 = img2_w.sum(axis=2) > 0
    mask3 = img3_w.sum(axis=2) > 0

    panorama = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    panorama[mask1] = img1_w[mask1]
    panorama[mask3] = img3_w[mask3]
    panorama[mask2] = img2_w[mask2]  # referencia tem prioridade

    return panorama, img1_w, img2_w, img3_w


panorama, img1_w, img2_w, img3_w = criar_panorama(img1, img2, img3, H_1to2, H_3to2)
print('Panorama criado com sucesso!')

### Imagens reprojetadas individualmente

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img_w, titulo in zip(
        axes,
        [img1_w, img2_w, img3_w],
        ['Img 1 reprojetada', 'Img 2 (referencia)', 'Img 3 reprojetada']):
    ax.imshow(bgr2rgb(img_w))
    ax.set_title(titulo, fontsize=12)
    ax.axis('off')
plt.suptitle('Imagens individuais no canvas do panorama', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Panorama Final

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
ax.imshow(bgr2rgb(panorama))
ax.set_title('Panorama - Combinacao das 3 Imagens via Homografia',
             fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Resolucao do panorama: {panorama.shape[1]}x{panorama.shape[0]} px')

### Salvar o panorama

In [ ]:
cv2.imwrite('panorama_resultado.jpg', panorama)
print('Panorama salvo em panorama_resultado.jpg')

# No Colab, inicia o download automatico do resultado.
try:
    from google.colab import files
    files.download('panorama_resultado.jpg')
    print('Download iniciado.')
except ImportError:
    print('Fora do Colab: arquivo salvo no diretorio atual.')